# Inference using Python API

In this notebook, we will walk you through execution of an NLP (Task: MaskedLM) model on Qualcomm Cloud AI 100 device using Python API.

This notebook will introduce you to qaic python library which is used for inference on **Qualcomm Cloud AI 100**.

Let's dive right in and get started!

# Prerequisites

### Steps to install ```qaic``` library.
```shell
pip install /opt/qti-aic/dev/lib/x86_64/qaic-0.0.1-py3-none-any.whl
pip install -r requirements.txt
```

### Steps to setup and open this notebook. 

```shell
sudo /opt/qti-aic/dev/python/qaic-env/bin/pip install ipykernel
sudo jupyter kernelspec list
sudo /opt/qti-aic/dev/python/qaic-env/bin/python -m ipykernel install --user --name qaic-env --display-name "qaic-env"
sudo jupyter notebook --no-browser --port 8080 --allow-root
```


In [1]:
# Let's make sure the Python interpreter path is set properly.
import sys
sys.executable

'/opt/qti-aic/dev/python/qaic-env/bin/python'

# Overview 

Here is an overview of what we will cover in this demo.
1. **Setup**: We will begin by importing all the necessary libraries and importing all the required dependencies.
2. **torch-cpu inference**: Import the model, generate an input and run the model on CPU.
3. **ONNX conversion**: We will convert the pytorch model to onnx format. 
4. **Compilation**: Compile the model for Qualcomm Cloud AI 100.
5. **Creating a Session and setting up inputs**: Create a qaic session and prepare the models for qaic runtime.
6. **Inference on Qualcomm Cloud AI 100**: Run the model with qaic api and decoding the output example.
6. **Inference on Qualcomm Cloud AI 100**: Run the model with qaic-runner CLI tool. 

# 1. Setup
Import the necessary libraries.

We will import the pre-trained model from Hugging Face ```transformers``` library. 


In [2]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import sys
sys.path.append("/opt/qti-aic/examples/apps/qaic-python-sdk")
import qaic
import os
import torch
import onnx
from onnxsim import simplify
import argparse
import numpy as np
from typing import Dict, List, Optional, Tuple

/opt/qti-aic/dev/python/qaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose a model from ```transformers``` library 
For example: you can provide any pretrained models, but accordingly create ```<model_name>-config.yaml``` file containing compilation and execution options. 

In [3]:
model_card = 'bert-base-cased' # Provide a model name supported in transformers library.

In [4]:
# Import the pre-trained model
model = AutoModelForMaskedLM.from_pretrained(model_card)

# setup the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_card)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [5]:
# Sentence example
sentence = "The dog [MASK] on the mat."

In [6]:
def get_example_input(tokenizer):
    max_length = 128
    encodings = tokenizer(sentence, max_length=max_length, truncation=True, padding="max_length", return_tensors='pt')
    inputIds = encodings["input_ids"]
    attentionMask = encodings["attention_mask"]
    mask_token_index = torch.where(encodings['input_ids'] == tokenizer.mask_token_id)[1]
    return inputIds, attentionMask, mask_token_index, encodings

In [7]:
# get input example 
inputIds, attentionMask, mask_token_index, inputs = get_example_input(tokenizer)

# 2. Inference using torch-cpu

In [8]:
# Execute on the host 
with torch.no_grad():
    pt_outputs = model(**inputs)

In [9]:
# decode the masked tokens from the pt_outputs variable.

mask_token_logits = pt_outputs.logits[0, mask_token_index, :]
top_5_results = torch.topk(mask_token_logits, 5, dim=1)
print("Model output (top5) from torch-cpu:")
for i in range(5):
    idx = top_5_results.indices[0].tolist()[i]
    val = top_5_results.values[0].tolist()[i]
    word = tokenizer.decode([idx])
    print(f"{i+1} :(word={word}, index={idx}, logit={round(val,2)})")

output_names = list(pt_outputs.keys())

Model output (top5) from torch-cpu:
1 :(word=sat, index=2068, logit=11.45)
2 :(word=landed, index=4860, logit=11.1)
3 :(word=spat, index=15732, logit=10.94)
4 :(word=settled, index=3035, logit=10.82)
5 :(word=was, index=1108, logit=10.75)


# 3. ONNX conversion

In [10]:
# Setup dir for saving onnx and qpc files
gen_models_path = f"{model_card}/generatedModels"
os.makedirs(gen_models_path, exist_ok=True)
model_base_name = model_card


In [11]:
# Set dynamic dims and axes.
dynamic_dims = {0: 'batch', 1 : 'sequence'}
dynamic_axes = {
    "input_ids" : dynamic_dims,
    "attention_mask" : dynamic_dims,
    "logits" : dynamic_dims
}
input_names = ["input_ids", "attention_mask"]
inputList = [inputIds, attentionMask]

model.eval() # setup the model in inference model.

torch.onnx.export(
    model,
    args=tuple(inputList),
    f=f"{gen_models_path}/{model_base_name}.onnx",
    verbose=False,
    input_names=input_names,
    output_names=["logits"],
    dynamic_axes=dynamic_axes,
    opset_version=11,
)
print("INFO: ONNX Model is being generated successfully")

INFO: ONNX Model is being generated successfully


#### Modification 
Modify the onnx file to handle ```constants > FP16_Max and < FP16_Min ```. 
```fix_onnx_fp16``` is a helper function for this purpose.

In [12]:
from onnx import numpy_helper
        
def fix_onnx_fp16(
    gen_models_path: str,
    model_base_name: str,
) -> str:
    finfo = np.finfo(np.float16)
    fp16_max = finfo.max
    fp16_min = finfo.min

    model = onnx.load(f"{gen_models_path}/{model_base_name}.onnx")
    fp16_fix = False
    for tensor in onnx.external_data_helper._get_all_tensors(model):
        nptensor = numpy_helper.to_array(tensor, gen_models_path)
        if nptensor.dtype == np.float32 and (
            np.any(nptensor > fp16_max) or np.any(nptensor < fp16_min)
        ):
            # print(f'tensor value : {nptensor} above {fp16_max} or below {fp16_min}')
            nptensor = np.clip(nptensor, fp16_min, fp16_max)
            new_tensor = numpy_helper.from_array(nptensor, tensor.name)
            tensor.CopyFrom(new_tensor)
            fp16_fix = True
            
    if fp16_fix:
        # Save FP16 model
        print("Found constants out of FP16 range, clipped to FP16 range")
        model_base_name += "_fix_outofrange_fp16"
        onnx.save(model, f=f"{gen_models_path}/{model_base_name}.onnx")
        print(f"Saving modified onnx file at {gen_models_path}/{model_base_name}.onnx")
    return model_base_name

fp16_model_name = fix_onnx_fp16(gen_models_path=gen_models_path, model_base_name=model_base_name)


Found constants out of FP16 range, clipped to FP16 range
Saving modified onnx file at bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx


# 4. Compilation step

`qaic-exec` cli tool is used to compile the model for Qualcomm AI Cloud 100. The input to this tool is `onnx` file generated above. The tool produces a QPC (Qualcomm Program Container) binary file in the path defined by `-aic-binary-dir` argument. 

### Breakdown of key compile parameters.
We have compiled the onnx file 
- with 4 NSP cores
- with float 16 precision
- defined onnx symbols


In [13]:
# COMPILE using qaic-exec
os.system('rm -fr bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc')

cmd = '/opt/qti-aic/exec/qaic-exec \
-m=bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx \
-aic-num-cores=4 \
-convert-to-fp16 \
-onnx-define-symbol=batch,1 -onnx-define-symbol=sequence,128 \
-aic-binary-dir=bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc \
-aic-hw -aic-hw-version=2.0 \
-compile-only'

os.system(cmd)

Reading ONNX Model from bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16.onnx
Compile started ............... 
Compiling model with FP16 precision.
Generated binary is present at bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc


0

## NOTE:

There are three different approaches to invoke the device for inference. 

1. Utilizing a command line inferface (CLI) command - ```qaic-runner```
2. Employing `Python` API (as shown below)
3. Leveraging the `C++` api.

# 5. Creating a Session and setting up inputs

Now the we have compiled the model for Qualcomm Cloud AI 100, we can setup a session to run the inference on the device. ```qaic``` library is a set of APIs that provides support for running inference on AIC100 backend. 

```Session```: Session is the entry point of these APIs. Session is a factory method which user needs to call to create an instance of session with AIC100 backend.

### API:
```Session(model_qpc_path: str, **kwargs)```


### Examples:
Creating Session with options passed as KW args
```python
sess = qaic.Session(model_path='/path/to/qpc', num_activations = 8, set_size=10) 
```
 
Creating a Session by passing options in yaml file
```python
sess = qaic.Session(model_path='/path/to/qpc', options_path = ‘/path/xyz/options.yaml’)
```


### **Limitations**
- APIs are compatible with only python 3.8 
- These APIs are supported only on x86 platforms

Lets create a bert session 
`bert-base-cased-config.yaml` contains inference parameters like num_activations which is used by qaic.Session 
along with input data for inference on the Cloud AI device.


In [14]:
# Contents of our yaml
options_path = f'{model_card}-config.yaml'
_ = os.system(f'cat {options_path}')

# Inference Parameters
num_activations: 2
set_size: 10

In [15]:
# Set the path of QPC generated with qaic-exec
qpcPath = 'bert-base-cased/generatedModels/bert-base-cased_fix_outofrange_fp16_qpc'

In [16]:

bert_sess = qaic.Session(model_path= qpcPath+'/programqpc.bin', options_path=options_path)

# alternatively, you can also provide arguments in the function call.
# bert_sess = qaic.Session(model_path= qpcPath+'/programqpc.bin', num_activations=2)

Here we are setting `num_activations = 1` and `set_size = 1`.
Additionally, you can provide `device_id` as inference parameters. 

Please find more details about the options [here](https://docs.qualcomm.com/bundle/resource/topics/AIC_Developer_Guide/).

In [17]:
# Here we are reading out all the input and output shapes/types
input_shape, input_type = bert_sess.model_input_shape_dict['input_ids']
attn_shape, attn_type = bert_sess.model_input_shape_dict['attention_mask']
output_shape, output_type = bert_sess.model_output_shape_dict['logits']
print(f'Input token shape {input_shape} and type {input_type}')
print(f'Input attention mask shape {attn_shape} and type {attn_type}')
print(f'Output logits shape {output_shape} and type {output_type}')

Input token shape (1, 128) and type int64
Input attention mask shape (1, 128) and type int64
Output logits shape (1, 128, 28996) and type float32


# 6. Inference on Qualcomm Cloud AI 100

In [18]:
# Create a input dictionary for given input.
input_dict = {"input_ids": inputIds.numpy().astype(input_type), "attention_mask" : attentionMask.numpy().astype(attn_type)}

In [19]:
# Run the model on Qualcomm Cloud AI 100
output = bert_sess.run(input_dict)

qaic: WARNING: Acitvating network, this will add up to time of first inference


In [20]:
# Restructure the data from output buffer with output_shape, output_type
token_logits = np.frombuffer(output['logits'], dtype=output_type).reshape(output_shape)

# Decode the output.
mask_token_logits = torch.from_numpy(token_logits[0, mask_token_index, :]).unsqueeze(0)
top_5_results = torch.topk(mask_token_logits, 5, dim=1)
print("Model output (top5) from Qualcomm Cloud AI 100:")
for i in range(5):
    idx = top_5_results.indices[0].tolist()[i]
    val = top_5_results.values[0].tolist()[i]
    word = tokenizer.decode([idx])
    print(f"{i+1} :(word={word}, index={idx}, logit={round(val,2)})")


Model output (top5) from Qualcomm Cloud AI 100:
1 :(word=sat, index=2068, logit=11.46)
2 :(word=landed, index=4860, logit=11.11)
3 :(word=spat, index=15732, logit=10.95)
4 :(word=settled, index=3035, logit=10.84)
5 :(word=was, index=1108, logit=10.75)


In [21]:
bert_sess.reset()

# 7. Inference using `qaic-runner` CLI tool

## a. generate inputs in a .raw file

In [22]:
# get example input
inputIds, attentionMask, mask_token_index, inputs = get_example_input(tokenizer)

# store the example input in files
import os

def save_encodings(inputIds, attentionMask, path):
    os.makedirs(path, exist_ok=True)
    inputIds.detach().numpy().tofile(os.path.join(path, 'input_ids.raw'))
    attentionMask.detach().numpy().tofile(os.path.join(path, 'input_mask.raw'))
    print('Encodings saved at:', path)

save_encodings(inputIds, attentionMask, path='bert-base-cased/inputFiles')



Encodings saved at: bert-base-cased/inputFiles


## b. call `qaic-runner` to run the inference

please note the inf/sec mentioned below is not with the best compile & exec parameters. 

In this case num_activations = 1 and set_size = 10 (default).

In [23]:
cmd = f'/opt/qti-aic/exec/qaic-runner \
    -t {qpcPath} \
    -a 1 \
    -n 1000 \
    -i bert-base-cased/inputFiles/input_ids.raw \
    -i bert-base-cased/inputFiles/input_mask.raw \
    --write-output-start-iter 1 \
    --write-output-num-samples 1 \
    -d 0'
os.system(cmd)

Input file: bert-base-cased/inputFiles/input_ids.raw
Input file: bert-base-cased/inputFiles/input_mask.raw
Writing file:./logits-activation-0-inf-1.bin
 ---- Stats ----
InferenceCnt 1000 TotalDuration 3854325us BatchSize 1 Inf/Sec 259.449


0

## c. read back and decode the outputs

In [24]:
# Output logits shape (1, 128, 28996) and type float32
output_shape = (1, 128, 28996)

token_logits = np.fromfile('./logits-activation-0-inf-1.bin', output_type).reshape(output_shape)
# Decode the output.
mask_token_logits = torch.from_numpy(token_logits[0, mask_token_index, :]).unsqueeze(0)
top_5_results = torch.topk(mask_token_logits, 5, dim=1)
print("Model output (top5) from Qualcomm Cloud AI 100:")
for i in range(5):
    idx = top_5_results.indices[0].tolist()[i]
    val = top_5_results.values[0].tolist()[i]
    word = tokenizer.decode([idx])
    print(f"{i+1} :(word={word}, index={idx}, logit={round(val,2)})")

Model output (top5) from Qualcomm Cloud AI 100:
1 :(word=sat, index=2068, logit=11.46)
2 :(word=landed, index=4860, logit=11.11)
3 :(word=spat, index=15732, logit=10.95)
4 :(word=settled, index=3035, logit=10.84)
5 :(word=was, index=1108, logit=10.75)
